# Census table download

Downloads ACS 5-year tables from the Census API: block-group tables for every Hawaiʻi block group and the Hawaiian Home Lands table (S0601) for every Hawaiian Home Land area.

Needs two settings in the `.env` file in the repo's top folder: `CENSUS_API_KEY` (your Census API key) and `SCOVI_PROJECT_DIR` (the Nextcloud "SCOVI Project" folder, whose `census/JPL` raw census files the downloads are compared with).

In [1]:
import csv
import glob
import json
import os
import urllib.parse
import urllib.request

from dotenv import load_dotenv

load_dotenv()

# Tables and ACS 5-year release to download, each with the folder it belongs in
TABLES = [
    "B25034",  # age_of_structure
    "B27010",  # health_insurance
    "B28010",  # households_w_computer
    "B28011",  # internet_subscription
    "C16002",  # limited_english_speaking
    # "B09019",  # living_arrangements: no block-group values after 2022
    "C17002",  # income_share_of_fpl
    "B01001",  # person_under_5_65
    "B02001",  # race_origin
    "B25003",  # tenure
    "B25014",  # tenure_by_occupants_per_room
    "B11003",  # family_type_by_children
    "B25044",  # tenure_by_vehicles
    "B23025",  # employment_status
    "B25070",  # renter_cost_burden
    "B25091",  # owner_cost_burden
    # "B25046", # aggregate_vehicles: no longer used
    # "B19001", # household_income: no longer used
]
YEAR = 2024

OUTPUT_DIR = os.path.expanduser("~/Downloads")

# The raw census files path to Nextcloud, only use for comparison, must be manually deposited there after download
RAW_DIR = os.path.join(os.path.expanduser(os.environ["SCOVI_PROJECT_DIR"]), "Metrics/All Exposures/census/JPL")

# Free key from api.census.gov/data/key_signup.html
API_KEY = os.environ["CENSUS_API_KEY"]

HAWAII_FIPS = "15"

In [2]:
# One request to the ACS 5-year API, as parsed JSON
def census_api_get(year, path, params):
    query = urllib.parse.urlencode(params + [("key", API_KEY)])
    with urllib.request.urlopen(f"https://api.census.gov/data/{year}/acs/acs5{path}?{query}") as response:
        return json.load(response)


# A cell as data.census.gov shows it: the API's note ("-", "**", "(X)"...) where it has one, 
# "null" where there is no value, else the value
def website_cell(value, note):
    if note is not None:
        return note
    if value is None:
        return "null"
    return str(value)


# An API answer laid out like a website download: the code row, the label row, then one row per place in place id order
def website_table(table, variables, response):
    records = sorted((dict(zip(response[0], row)) for row in response[1:]), key=lambda record: record["GEO_ID"])

    estimates = sorted(code for code in variables if code.startswith(f"{table}_") and code.endswith("E"))
    codes = ["GEO_ID", "NAME"] + [code for estimate in estimates for code in (estimate, estimate[:-1] + "M")]
    labels = [variables[code]["label"] for code in codes]
    rows = [[website_cell(record[code], record.get(code + "A")) for code in codes] for record in records]
    return [codes, labels] + rows


# A detailed table for every Hawaiʻi block group
def download_block_group_table(table, year):
    variables = census_api_get(year, f"/groups/{table}.json", [])["variables"]
    response = census_api_get(year, "", [
        ("get", f"group({table})"),
        ("for", "block group:*"),
        ("in", f"state:{HAWAII_FIPS}"),
        ("in", "county:*"),
        ("in", "tract:*"),
    ])
    return website_table(table, variables, response)


# A subject table for every Hawaiian Homeland area. 
# API publishes these areas for whole country, not by state, so the Hawaiian Homelands must be specified
def download_homelands_table(table, year):
    variables = census_api_get(year, f"/subject/groups/{table}.json", [])["variables"]
    response = census_api_get(year, "/subject", [
        ("get", f"group({table})"),
        ("for", "american indian area/alaska native area/hawaiian home land:*"),
    ])
    rows = website_table(table, variables, response)
    return rows[:2] + [row for row in rows[2:] if "Hawaiian Home Land" in row[1]]


# Saves rows as <file_prefix><year>.<table>-Data.csv (ACSDT5Y for detailed tables, ACSST5Y for subject tables),
# the name the pipeline reads the product, table and years from
def save_census_csv(rows, file_prefix, table, year, output_dir):
    path = os.path.join(output_dir, f"{file_prefix}{year}.{table}-Data.csv")
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerows(rows)
    return path


# A raw census file's rows, without blank lines or the empty last column some website downloads end each row with
def read_census_csv(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        rows = [row for row in csv.reader(f) if row]
    if rows[0][-1] == "":
        rows = [row[:-1] for row in rows]
    return rows


# Compares a download with the table's current version being used: codes, labels and places
def compare_with_current(table, rows, file_prefix):
    current_paths = sorted(glob.glob(os.path.join(RAW_DIR, "*", f"{file_prefix}*.{table}-Data.csv")))
    if not current_paths:
        print(f"{table}: no current file in the JPL folders to compare with")

    new_labels = dict(zip(rows[0], rows[1]))
    new_places = sorted(row[0] for row in rows[2:])

    for current_path in current_paths:
        current = read_census_csv(current_path)
        current_labels = dict(zip(current[0], current[1]))

        print(f"{table}: {os.path.relpath(current_path, RAW_DIR)}")
        print("  Same codes:", current[0] == rows[0])
        print("  Same labels:", current[1] == rows[1])
        print("  Same places:", sorted(row[0] for row in current[2:]) == new_places)
        print("  Codes only in the current file:", sorted(set(current_labels) - set(new_labels)))
        print("  Codes only in the download:", sorted(set(new_labels) - set(current_labels)))
        print("  Relabelled:", [
            (code, current_labels[code], new_labels[code])
            for code in current_labels
            if code in new_labels and current_labels[code] != new_labels[code]
        ])

## Download

In [ ]:
# Every ACS block-group file covers the same block groups, so each download must match each of them
raw_paths = sorted(glob.glob(os.path.join(RAW_DIR, "*", "ACSDT5Y*-Data.csv")))
assert raw_paths, "No raw census files in the SCOVI_PROJECT_DIR census/JPL folder"
raw_block_groups = {
    os.path.relpath(raw_path, RAW_DIR): sorted(row[0] for row in read_census_csv(raw_path)[2:])
    for raw_path in raw_paths
}

downloads = {}
for table in TABLES:
    rows = download_block_group_table(table, YEAR)
    path = save_census_csv(rows, "ACSDT5Y", table, YEAR, OUTPUT_DIR)
    block_groups = sorted(row[0] for row in rows[2:])
    for raw_path, raw_groups in raw_block_groups.items():
        assert raw_groups == block_groups, f"{table}: block groups differ from {raw_path}"
    downloads[table] = rows
    print(f"Saved {len(block_groups)} block groups to {os.path.basename(path)}")

## Compare with the current files

For each downloaded table, finds its current file in the JPL folders and checks the download has the same column codes, labels and places. Same labels means the JPL spreadsheet and the menu still fit the new file.

In [ ]:
for table, rows in downloads.items():
    compare_with_current(table, rows, "ACSDT5Y")

## Hawaiian Home Lands

Downloads the Homelands table, S0601 ("Selected Characteristics of the Total and Native Populations in the United States"), for every Hawaiian Home Land area. It's a subject table published for Home Land areas rather than block groups. Saves `ACSST5Y<year>.S0601-Data.csv` to Downloads and compares it with the current Homelands file in the JPL folders.

In [3]:
# Subject table and ACS 5-year release to download for the Hawaiian Home Lands
HOMELANDS_TABLE = "S0601"
HOMELANDS_YEAR = 2024

homelands = download_homelands_table(HOMELANDS_TABLE, HOMELANDS_YEAR)
path = save_census_csv(homelands, "ACSST5Y", HOMELANDS_TABLE, HOMELANDS_YEAR, OUTPUT_DIR)
print(f"Saved {len(homelands) - 2} Hawaiian Home Lands to {os.path.basename(path)}")

compare_with_current(HOMELANDS_TABLE, homelands, "ACSST5Y")

Saved 74 Hawaiian Home Lands to ACSST5Y2024.S0601-Data.csv
S0601: 2022_census_hawaiian_homelands/ACSST5Y2022.S0601-Data.csv
  Same codes: True
  Same labels: False
  Same places: False
  Codes only in the current file: []
  Codes only in the download: []
  Relabelled: [('S0601_C01_025E', 'Estimate!!Total!!LANGUAGE SPOKEN AT HOME AND ABILITY TO SPEAK ENGLISH!!Population 5 years and over!!Speak language other than English!!Speak English  very well""', 'Estimate!!Total!!LANGUAGE SPOKEN AT HOME AND ABILITY TO SPEAK ENGLISH!!Population 5 years and over!!Speak language other than English!!Speak English  "very well"'), ('S0601_C01_025M', 'Margin of Error!!Total!!LANGUAGE SPOKEN AT HOME AND ABILITY TO SPEAK ENGLISH!!Population 5 years and over!!Speak language other than English!!Speak English  very well""', 'Margin of Error!!Total!!LANGUAGE SPOKEN AT HOME AND ABILITY TO SPEAK ENGLISH!!Population 5 years and over!!Speak language other than English!!Speak English  "very well"'), ('S0601_C01_026E